In [7]:
import os
import joblib
import librosa
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
MODEL_PATH = '../models/rf_model.pkl'
LABEL_ENCODER_PATH = '../models/rf_label_encoder.pkl'
SPOTIFY_CSV = '../data/dataset.csv'

required_files = [MODEL_PATH, LABEL_ENCODER_PATH, SPOTIFY_CSV]

missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

model = joblib.load(MODEL_PATH)
le = joblib.load(LABEL_ENCODER_PATH)

df = pd.read_csv(SPOTIFY_CSV)
AUDIO_FEATURES = [
    'valence',
    'energy',
    'tempo',
    'danceability',
    'acousticness',
    'instrumentalness',
    'loudness'
]

df = df[['track_name', 'artists', 'track_genre'] + AUDIO_FEATURES].dropna()
df = df.drop_duplicates(subset=['track_name', 'artists']).reset_index(drop=True)

for col in ['tempo', 'loudness']:
    col_min = df[col].min()
    col_max = df[col].max()
    if col_max != col_min:
        df[col] = (df[col] - col_min) / (col_max - col_min)
    else:
        df[col] = 0.0




In [13]:
EMOTION_PROFILES = {
    'happy': {
        'targets': {'valence': 0.85, 'energy': 0.80, 'tempo': 0.72, 'danceability': 0.82,
                    'acousticness': 0.20, 'instrumentalness': 0.08, 'loudness': 0.72},
        'genre_keywords': ['pop', 'dance', 'disco', 'funk', 'samba', 'salsa', 'house'],
        'filters': {'valence_min': 0.60, 'energy_min': 0.58, 'danceability_min': 0.55},
        'weights': {'similarity': 0.45, 'valence': 0.20, 'energy': 0.15,
                    'danceability': 0.15, 'tempo': 0.05}
    },
    'sad': {
        'targets': {'valence': 0.20, 'energy': 0.25, 'tempo': 0.30, 'danceability': 0.30,
                    'acousticness': 0.70, 'instrumentalness': 0.30, 'loudness': 0.30},
        'genre_keywords': ['acoustic', 'ambient', 'piano', 'indie', 'chill'],
        'filters': {'valence_max': 0.42, 'energy_max': 0.45},
        'weights': {'similarity': 0.45, 'acousticness': 0.20, 'instrumentalness': 0.10,
                    'valence_inverse': 0.15, 'energy_inverse': 0.10, 'tempo_inverse': 0.10}
    },
    'angry': {
        'targets': {'valence': 0.30, 'energy': 0.95, 'tempo': 0.86, 'danceability': 0.58,
                    'acousticness': 0.10, 'instrumentalness': 0.15, 'loudness': 0.90},
        'genre_keywords': ['rock', 'metal', 'punk', 'hardcore', 'grunge'],
        'filters': {'energy_min': 0.72, 'loudness_min': 0.55},
        'weights': {'similarity': 0.50, 'energy': 0.20, 'loudness': 0.15,
                    'tempo': 0.10, 'danceability': 0.05}
    },
    'fearful': {
        'targets': {'valence': 0.25, 'energy': 0.55, 'tempo': 0.50, 'danceability': 0.35,
                    'acousticness': 0.45, 'instrumentalness': 0.35, 'loudness': 0.45},
        'genre_keywords': ['ambient', 'dark', 'soundtrack', 'electronic'],
        'filters': {'valence_max': 0.45},
        'weights': {'similarity': 0.50, 'instrumentalness': 0.15, 'acousticness': 0.10,
                    'energy': 0.10, 'tempo': 0.05, 'loudness': 0.10}
    },
    'neutral': {
        'targets': {'valence': 0.50, 'energy': 0.50, 'tempo': 0.50, 'danceability': 0.50,
                    'acousticness': 0.40, 'instrumentalness': 0.25, 'loudness': 0.50},
        'genre_keywords': [],
        'filters': {},
        'weights': {'similarity': 0.60, 'valence_balance': 0.10,
                    'energy_balance': 0.10, 'danceability': 0.10, 'tempo': 0.10}
    },
    'calm': {
        'targets': {'valence': 0.60, 'energy': 0.25, 'tempo': 0.30, 'danceability': 0.40,
                    'acousticness': 0.78, 'instrumentalness': 0.50, 'loudness': 0.25},
        'genre_keywords': ['ambient', 'classical', 'jazz', 'chill'],
        'filters': {'energy_max': 0.40, 'tempo_max': 0.45},
        'weights': {'similarity': 0.45, 'acousticness': 0.20, 'instrumentalness': 0.15,
                    'energy_inverse': 0.10, 'tempo_inverse': 0.10}
    },
    'surprised': {
        'targets': {'valence': 0.72, 'energy': 0.78, 'tempo': 0.68, 'danceability': 0.68,
                    'acousticness': 0.18, 'instrumentalness': 0.10, 'loudness': 0.72},
        'genre_keywords': ['pop', 'edm', 'electro', 'funk'],
        'filters': {'energy_min': 0.55},
        'weights': {'similarity': 0.45, 'energy': 0.18, 'valence': 0.15,
                    'danceability': 0.12, 'tempo': 0.10}
    }
}


In [14]:
def extract_features(filepath, sr=22050, duration=3.0):
    y, sr = librosa.load(filepath, sr=sr, duration=duration)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc_feat = np.hstack([mfcc.mean(axis=1), mfcc.std(axis=1)])

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_feat = np.hstack([chroma.mean(axis=1), chroma.std(axis=1)])

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_feat = np.array([centroid.mean(), centroid.std()])

    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_feat = np.array([zcr.mean(), zcr.std()])

    rms = librosa.feature.rms(y=y)
    rms_feat = np.array([rms.mean(), rms.std()])

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    rolloff_feat = np.array([rolloff.mean(), rolloff.std()])

    return np.hstack([
        mfcc_feat,
        chroma_feat,
        centroid_feat,
        zcr_feat,
        rms_feat,
        rolloff_feat
    ])


In [15]:
def predict_emotion(audio_path):
    feats = extract_features(audio_path).reshape(1, -1)

    pred_enc = model.predict(feats)[0]
    return le.inverse_transform([pred_enc])[0]



In [16]:
def apply_emotion_filters(candidates, emotion):
    rules = EMOTION_PROFILES[emotion]['filters']
    filtered = candidates.copy()

    if 'valence_min' in rules:
        filtered = filtered[filtered['valence'] >= rules['valence_min']]
    if 'valence_max' in rules:
        filtered = filtered[filtered['valence'] <= rules['valence_max']]
    if 'energy_min' in rules:
        filtered = filtered[filtered['energy'] >= rules['energy_min']]
    if 'energy_max' in rules:
        filtered = filtered[filtered['energy'] <= rules['energy_max']]
    if 'danceability_min' in rules:
        filtered = filtered[filtered['danceability'] >= rules['danceability_min']]
    if 'tempo_max' in rules:
        filtered = filtered[filtered['tempo'] <= rules['tempo_max']]
    if 'loudness_min' in rules:
        filtered = filtered[filtered['loudness'] >= rules['loudness_min']]

    return filtered if len(filtered) >= 20 else candidates


In [17]:
def apply_genre_filter(candidates, emotion):
    keywords = EMOTION_PROFILES[emotion]['genre_keywords']
    if not keywords:
        return candidates

    mask = candidates['track_genre'].astype(str).str.lower().apply(
        lambda g: any(k in g for k in keywords)
    )

    genre_filtered = candidates[mask]
    return genre_filtered if len(genre_filtered) >= 20 else candidates

In [18]:
def compute_final_score(candidates, emotion):
    weights = EMOTION_PROFILES[emotion]['weights']
    scored = candidates.copy()
    scored['final_score'] = 0.0

    for k, w in weights.items():
        if k == 'similarity':
            scored['final_score'] += w * scored['similarity_score']
        elif k == 'valence_inverse':
            scored['final_score'] += w * (1 - scored['valence'])
        elif k == 'energy_inverse':
            scored['final_score'] += w * (1 - scored['energy'])
        elif k == 'tempo_inverse':
            scored['final_score'] += w * (1 - scored['tempo'])
        elif k == 'valence_balance':
            scored['final_score'] += w * (1 - (scored['valence'] - 0.5).abs() * 2)
        elif k == 'energy_balance':
            scored['final_score'] += w * (1 - (scored['energy'] - 0.5).abs() * 2)
        else:
            scored['final_score'] += w * scored[k]

    return scored.sort_values('final_score', ascending=False)


In [19]:
def diversify_recommendations(candidates, top_n=10, max_per_genre=2, max_per_artist=1):
    final = []
    genre_count = {}
    artist_count = {}

    for _, row in candidates.iterrows():
        g = str(row['track_genre']).lower()
        a = str(row['artists']).lower()

        if genre_count.get(g, 0) >= max_per_genre:
            continue
        if artist_count.get(a, 0) >= max_per_artist:
            continue

        final.append(row)
        genre_count[g] = genre_count.get(g, 0) + 1
        artist_count[a] = artist_count.get(a, 0) + 1

        if len(final) == top_n:
            break

    return pd.DataFrame(final).reset_index(drop=True)

In [20]:
def recommend_songs(emotion, top_n=10):
    targets = EMOTION_PROFILES[emotion]['targets']

    query_vec = np.array([[targets[f] for f in AUDIO_FEATURES]])
    track_matrix = df[AUDIO_FEATURES].values

    similarity_scores = cosine_similarity(query_vec, track_matrix)[0]

    candidates = df.copy()
    candidates['similarity_score'] = similarity_scores

    candidates = apply_emotion_filters(candidates, emotion)
    candidates = apply_genre_filter(candidates, emotion)
    candidates = compute_final_score(candidates, emotion)
    candidates = diversify_recommendations(candidates, top_n)

    return candidates[[
        'track_name', 'artists', 'track_genre',
        'similarity_score', 'final_score'
    ]]

In [21]:
def mood_music_pipeline(audio_path, top_n=10):
    print(f'Analyzing: {os.path.basename(audio_path)}')
    print('-' * 50)

    emotion = predict_emotion(audio_path)
    print(f'Detected Emotion: {emotion.upper()}')

    print(f'\nTop {top_n} songs:')
    print('-' * 50)

    recs = recommend_songs(emotion, top_n)
    print(recs.to_string(index=False))

    return emotion, recs

In [22]:
test_file = '../data/archive/Actor_01/03-01-03-01-01-01-01.wav'
emotion, recs = mood_music_pipeline(test_file, top_n=10)

Analyzing: 03-01-03-01-01-01-01.wav
--------------------------------------------------
Detected Emotion: HAPPY

Top 10 songs:
--------------------------------------------------
                       track_name                 artists   track_genre  similarity_score  final_score
                   The Israelites           Apache Indian     dancehall          0.993309     0.937428
           Combo De Boca (Tô Mal)             Leo Santana          funk          0.984057     0.937093
                Hop Hop (Maniury)                 MiłyPan         disco          0.987211     0.935834
                    Ra Ta Ta 2011        Mallorca Cowboys         disco          0.988147     0.934269
Freedom of Choice - 2009 Remaster                    DEVO     synth-pop          0.991377     0.932136
                           Azonto  Fuse ODG;Tiffany Owusu     dancehall          0.986929     0.930651
                Super Freaky Girl             Nicki Minaj         dance          0.986630     0.929860